In [3]:
import json
import pandas as pd
import networkx as nx
import re
import pickle
import ast
from collections import defaultdict

# Load data
G = pickle.load(open("../Pickle/kg.pkl", "rb"))

with open("../Pickle/cond_count.pkl", "rb") as f:
    cond_count = pickle.load(f)

with open("../Pickle/evid_count.pkl", "rb") as f:
    evid_count = pickle.load(f)

In [2]:
pattern = re.compile(r'^(?P<eid>E_\d+)_@_(?P<vid>V_\d+|\d+)$')

In [3]:
import math

SMOOTH = 0.01


def safe_log(p):
    return math.log(max(p, SMOOTH))


def get_top_conditions(scores, k=10):
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]


def get_discriminating_evidence(G, candidate_conditions, asked):
    best_e, best_var = None, -1.0

    for c in candidate_conditions:
        for _, e, _ in G.out_edges(c, data=True):
            if G.nodes[e]["type"] != "evidence" or e in asked:
                continue

            probs = [
                G.edges[c2, e]["p_e_given_c"] if G.has_edge(c2, e) else 0.0
                for c2 in candidate_conditions
            ]

            if len(probs) < 2:
                continue

            mean = sum(probs) / len(probs)
            var = sum((p - mean) ** 2 for p in probs) / len(probs)

            if var > best_var:
                best_var, best_e = var, e

    return best_e


def get_top_values(G, evidence, candidate_conditions, k=5):
    scores = []

    for _, v in G.out_edges(evidence):
        if not G.nodes[v]["type"].startswith("possible"):
            continue

        stats = G.edges[evidence, v].get("cond_stats", {})
        max_p = max(
            stats.get(c, {}).get("p_v_given_e_c", 0.0)
            for c in candidate_conditions
        )

        scores.append((v, max_p))

    scores.sort(key=lambda x: x[1], reverse=True)
    return [v for v, _ in scores[:k]]


def run_ddx_traversal(G, scores, max_steps=6, top_k_conditions=10, top_k_values=20):

    asked = set()

    print("\n=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===")

    for step in range(max_steps):

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_conditions]
        candidate_conditions = [c for c, _ in ranked]

        print(f"\nStep {step + 1} — Current Differential:")
        for c, s in ranked:
            print(f"  {c:40s} score={s:.4f}")

        if len(candidate_conditions) <= 1:
            break

        evidence = get_discriminating_evidence(G, candidate_conditions, asked)
        if evidence is None:
            break

        asked.add(evidence)
        
        print(f"\n🩺 Question: {G.nodes[evidence]['question_en']}")
        print(f"\nEvidence ID: {evidence}")
        values = [
            v for _, v in G.out_edges(evidence)
            if G.nodes[v]["type"].startswith("possible")
        ]

        # ---------------- Binary evidence ----------------
        if not values:
            ans = input("Answer (yes / no): ").strip().lower()
            if ans in ("yes", "y"):
                for c in scores:
                    p = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else 0.0
                    scores[c] += safe_log(p)
            continue

        # ---------------- Value-based evidence ----------------
        shown_values = get_top_values(G, evidence, candidate_conditions, top_k_values)

        print("\nOptions:")
        for i, v in enumerate(shown_values):
            print(f"  {i}: {G.nodes[v]['value']}")
        print(f"  {len(shown_values)}: Other / none")

        choice = input(f"Choose options (0-{len(shown_values)}, space separated): ")
        # display(choice)
        choices = choice.split()
        if not all(ch.isdigit() for ch in choices):
            continue

        choices = [int(ch) for ch in choices]
        if any(ch == len(shown_values) for ch in choices):
            continue
        if any(ch < 0 or ch >= len(shown_values) for ch in choices):
            continue

        chosen_values = [shown_values[ch] for ch in choices]

        for c in scores:
            p_e = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else 0.0
            for chosen_value in chosen_values:
                stats = G.edges[evidence, chosen_value].get("cond_stats", {})
                p_v = stats.get(c, {}).get("p_v_given_e_c", 0.0)
                scores[c] += safe_log(p_v)

            scores[c] += safe_log(p_e)

    print("\n=== FINAL RANKED CONDITIONS ===")
    for c, s in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(f"{c:40s} score={s:.4f}")



In [4]:
import math
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==============================================================================
# CONFIGURATION & CONSTANTS
# ==============================================================================
# The model name must match the one used to generate the embeddings in your
# Knowledge Graph builder. Using a different model will cause vector mismatch.
EMBEDDING_MODEL = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'

# Thresholds to control the sensitivity of the NLU engine.
# THRESH_EVIDENCE: (0.0 - 1.0)
#   - Minimum similarity score required to assume a user is talking about a
#     specific symptom (Evidence).
#   - Lowering this increases recall (catching subtle symptoms) but risks
#     false positives (matching noise).
THRESH_EVIDENCE = 0.50

# THRESH_VALUE: (0.0 - 1.0)
#   - Minimum similarity score required to confirm a specific detail/value
#     (e.g., matching "Chest" instead of just generic "Pain").
#   - This should be higher than THRESH_EVIDENCE to prevent the system from
#     hallucinating specific details when the user was actually vague.
THRESH_VALUE = 0.65

# ==============================================================================
# CLASS: DDxGraphNLU
# Responsibility: Acts as the semantic search engine for the DDxPlus Knowledge Graph.
# It maps unstructured natural language to structured Graph Nodes (Evidence & Values).
# ==============================================================================
class DDxGraphNLU:
    def __init__(self, G, embedding_model_name=EMBEDDING_MODEL):
        """
        Initialization Block.
        Responsibility:
        1. Load the heavy NLP model (SentenceTransformer).
        2. "Index" the Knowledge Graph by extracting pre-computed embeddings
           from Evidence nodes and stacking them into a matrix.

        Why? This converts the search problem from a slow graph traversal
        into a fast Matrix Multiplication operation.
        """
        self.G = G
        print(f"Loading Model: {embedding_model_name}...")
        self.model = SentenceTransformer(embedding_model_name)

        # --- INDEXING STAGE ---
        # We prepare a list to hold the ID and Embedding for every Evidence node.
        self.evidence_ids = []
        embeddings_list = []

        print("Loading embeddings from Graph Evidence Nodes...")
        count = 0

        # Iterate over all nodes in the NetworkX graph
        for node, data in G.nodes(data=True):
            # Filter for nodes that act as "Questions" (Evidences)
            if data.get("type") == "evidence":

                # OPTIMIZATION: Check if embedding exists in the graph.
                # This avoids re-calculating embeddings (which is slow) every time we restart.
                if "embedding" in data:
                    emb = np.array(data["embedding"])
                    embeddings_list.append(emb)
                    self.evidence_ids.append(node)
                    count += 1
                else:
                    # Fallback: If the graph builder forgot to embed this node,
                    # we do it now on-the-fly (slower startup).
                    txt = f"{data.get('question_en', '')} {data.get('question_fr', '')}"
                    emb = self.model.encode(txt)
                    embeddings_list.append(emb)
                    self.evidence_ids.append(node)

        # Convert list of vectors into a 2D Numpy Matrix for vectorized cosine_similarity
        if count > 0:
            self.evidence_matrix = np.vstack(embeddings_list)
            print(f" -> Loaded {len(self.evidence_ids)} evidence vectors.")
        else:
            print("Warning: No pre-computed embeddings found. Engine might be slow.")

    def parse_query(self, user_query):
        """
        Main Processing Pipeline.
        Responsibility:
        1. Breaks complex sentences into atomic "Chunks" (Span Chunking).
        2. Detects negation logic (e.g., "no fever").
        3. Orchestrates the search for each chunk and aggregates findings.

        Args:
            user_query (str): The raw input string from the user.

        Returns:
            list: A list of result dictionaries containing 'eid', 'value', 'negated', etc.
        """
        all_findings = []

        # --- STEP 1: SPAN CHUNKING ---
        # We split the user's text into logical units based on punctuation and conjunctions.
        # The Regex `(?<!\d)\.(?!\d)` splits on periods but PROTECTS decimals (e.g., "39.5").
        delimiters = r'[,;]|\bbut\b|\band\b|\balso\b|\bhowever\b|\bplus\b|\bexcept\b|(?<!\d)\.(?!\d)'
        raw_chunks = [c.strip() for c in re.split(delimiters, user_query.lower()) if c.strip()]

        print(f"\n[NLU] Processing: '{user_query}'")

        for i, chunk in enumerate(raw_chunks):
            # # Skip very short chunks that are likely noise
            # if len(chunk) < 3: continue
            print(f" -> Chunk {i+1}: '{chunk}'")

            # --- STEP 2: NEGATION CHECK ---
            # Simple heuristic to see if the chunk contains negation words.
            # If true, we flag the result so the Diagnosis Engine knows to treat it as "Absent".
            is_negated = any(n in chunk for n in ["no ", "not ", "don't ", "never ", "without "])

            # --- STEP 3: SEMANTIC SEARCH ---
            # Run the core matching logic on this specific chunk
            matches = self._find_best_match(chunk)

            if matches:
                for match in matches:
                    if is_negated: match['negated'] = True
                    all_findings.append(match)

        evidences = []
        values = []
        # pattern = re.compile(r'^(?P<eid>E_\d+)_@_(?P<vid>V_\d+|\d+)$')

        for res in all_findings:
            evidences.append(res['eid'])
            status = "NO" if res.get('negated') else "YES"
            if res['value'] and status == "YES":
                # match = pattern.match(res['value'])
                # if match:
                values.append(res['value'])
            else:
                values.append(status)

        return all_findings, evidences, values

    def _find_best_match(self, text, top_k=5):
            query_vec = self.model.encode([text])
            scores = cosine_similarity(query_vec, self.evidence_matrix)[0]

            # Get Top-K Candidates
            top_indices = np.argsort(scores)[::-1][:top_k]

            best_results = []
            best_global_score = -1.0

            print(f"    [Debug] Checking Top {top_k} Candidates for span: '{text}'")

            for idx in top_indices:
                eid = self.evidence_ids[idx]
                e_score = scores[idx]

                evidence_node = self.G.nodes[eid]
                dtype = evidence_node.get("data_type", "B")

                # Initialize candidate with the Question Score
                current_candidate = {
                    'eid': eid,
                    'value': [],
                    'data_type': dtype,
                    'score': e_score,
                    'match_type': 'question_match'
                }

                # --- STAGE 2: VALUE CHECK (The Fix) ---
                # Check values BEFORE applying the threshold filter.
                # This allows low-scoring questions to be saved by high-scoring answers.
                if dtype in ["M", "C"]:
                    val_nodes = [v for _, v in self.G.out_edges(eid)
                                if self.G.nodes[v].get("type", "").startswith("possible")]

                    if val_nodes:
                        val_ids = []
                        val_embs = []
                        for v in val_nodes:
                            v_data = self.G.nodes[v]
                            if "embedding" in v_data:
                                val_ids.append(v)
                                val_embs.append(np.array(v_data["embedding"]))

                        if val_embs:
                            val_matrix = np.vstack(val_embs)
                            v_scores = cosine_similarity(query_vec, val_matrix)[0]
                            for i, vs in enumerate(v_scores): 
                                # BOOST SCORE if value is better
                                if vs > current_candidate['score']:
                                    current_candidate['score'] = vs
                                    current_candidate['match_type'] = 'value_match'

                                    # Only lock in the value if it's confident enough
                                    if vs >= THRESH_VALUE:
                                        current_candidate['value'].append(val_ids[i])

                # --- LATE FILTERING ---
                # NOW we check if the (potentially boosted) score meets the threshold
                if current_candidate['score'] < THRESH_EVIDENCE:
                    # Debug print to see what we skipped
                    # print(f"      -> Skipped {eid} (Score {current_candidate['score']:.3f} < {THRESH_EVIDENCE})")
                    continue

                print(f"      -> Candidate {eid}: RawQ={e_score:.3f}, FinalScore={current_candidate['score']:.3f}")

                best_results.append(current_candidate)
                if current_candidate['score'] > best_global_score:
                    best_global_score = current_candidate['score']

            sorted_results = sorted(best_results, key=lambda x: x['score'], reverse=True)
            k = 0
            for res in sorted_results:
                if best_global_score - res['score'] > 0.05:
                    break
                k += 1
            
            return sorted_results[:k] if k > 0 else None


# ==============================================================================
# MAIN EXECUTION BLOCK (Usage Example)
# ==============================================================================
if __name__ == "__main__":
    # Assumes 'G' is a NetworkX graph already loaded with DDxPLUS data
    # and embeddings on the nodes.

    # 1. Initialize Engine
    nlu = DDxGraphNLU(G)

    # 2. Define a complex user query with multiple symptoms and negation
    user_input = "I have no pain"

    # 3. Run the Parser
    results, evidence, values = nlu.parse_query(user_input)

    # 4. Display Results
    print(f"\n=== RESULT: Found {len(results)} items ===")
    for res in results:
        status = "NO" if res.get('negated') else "YES"
        if res['value']:
            val_str = "= "
            for v in res['value']:
                val_str += f"{v}, "
            val_str = val_str.rstrip(", ")
        else:
            val_str = ""
        print(f"Evidence: {res['eid']}{val_str} [{status}]")

    for evid, val in zip(evidence, values):
        print(f"{evid}: {val}")

Loading Model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext...


No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Loading embeddings from Graph Evidence Nodes...
 -> Loaded 223 evidence vectors.

[NLU] Processing: 'I have no pain'
 -> Chunk 1: 'i have no pain'
    [Debug] Checking Top 5 Candidates for span: 'i have no pain'
      -> Candidate E_55: RawQ=0.783, FinalScore=0.783
      -> Candidate E_56: RawQ=0.738, FinalScore=0.738
      -> Candidate E_54: RawQ=0.580, FinalScore=0.749
      -> Candidate E_216: RawQ=0.576, FinalScore=0.576
      -> Candidate E_58: RawQ=0.557, FinalScore=0.557

=== RESULT: Found 3 items ===
Evidence: E_55 [NO]
Evidence: E_54= E_54_@_V_11 [NO]
Evidence: E_56 [NO]
E_55: NO
E_54: NO
E_56: NO


In [5]:
# Code Updated by Varun
# --- CONFIGURATION ---
SMOOTH = 1e-6         # Tiny value to prevent log(0) errors
MAX_DELTA = 2.0       # Caps the maximum score change per question to ensure stability
ABSENCE_PROB_THRESHOLD = 0.5  # Threshold to decide if a symptom's absence is "significant"
ABSENCE_WEIGHT = 0.5  # Multiplier for the penalty applied when a 'expected' symptom is missing

def safe_log(p):
    """Calculates natural log while avoiding mathematical errors with zero."""
    return math.log(max(p, SMOOTH))

def capped_add(score, delta):
    """Updates a condition's score but prevents outliers from swinging the score too far."""
    return score + max(-MAX_DELTA, min(MAX_DELTA, delta))

def get_discriminating_evidence(G, candidate_conditions, scores, asked):
    """
    Selects the symptom that best distinguishes between the current top disease candidates.
    Uses a weighted variance approach (similar to Information Gain).
    """
    best_e, best_gain = None, -1.0

    # 1. Convert current logarithmic scores into a normalized probability distribution (Posteriors)
    max_s = max(scores[c] for c in candidate_conditions)
    post = {
        c: math.exp(scores[c] - max_s)
        for c in candidate_conditions
    }
    Z = sum(post.values())
    post = {c: p / Z for c, p in post.items()} # Ensure they sum to 1.0

    # 2. Iterate through all possible symptoms connected to the top candidates
    for c in candidate_conditions:
        for _, e, _ in G.out_edges(c, data=True):
            if G.nodes[e]["type"] != "evidence" or e in asked:
                continue

            # 3. Calculate how much 'disagreement' exists between candidates for this symptom
            ps = []
            for c2 in candidate_conditions:
                p = G.edges[c2, e]["p_e_given_c"] if G.has_edge(c2, e) else SMOOTH
                ps.append((post[c2], p))

            mean = sum(w * p for w, p in ps)
            gain = sum(w * (p - mean) ** 2 for w, p in ps)

            # 4. Keep the symptom that provides the most 'Information Gain'
            if gain > best_gain:
                best_gain, best_e = gain, e

    return best_e

def run_ddx_traversal_1(G, scores, max_steps=15, top_k_conditions=10, top_k_values=20, initial_asked=None, observed_yes=None, observed_no=None):
    """
    Conducts the interactive diagnostic session.
    """
    asked = set() if initial_asked is None else set(initial_asked)
    observed_yes = set() if observed_yes is None else set(observed_yes)
    observed_no = set() if observed_no is None else set(observed_no)
    nlu = DDxGraphNLU(G)

    print("\n=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===")

    for step in range(max_steps):
        # --- A. Rank and Display the current Top Candidates ---
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_conditions]
        candidate_conditions = [c for c, _ in ranked]

        print(f"\nStep {step + 1} — Current Differential:")
        for c, s in ranked:
            print(f"  {c:40s} score={s:.4f}")

        if len(candidate_conditions) <= 1:
            break

        # --- B. Decide the next Question ---
        evidence = get_discriminating_evidence(G, candidate_conditions, scores, asked)
        if evidence is None: break

        print(f"Selected evidence: {G.nodes[evidence]}")
        parent = G.nodes[evidence].get("parent", None)
        print(f"Parent evidence: {parent}")

        if parent and parent != evidence and parent not in observed_yes:
            if parent not in asked:
                print(f"\n🩺 Question: {G.nodes[parent]['question_en']}")
                print(f"\nEvidence ID: {parent}")

                ans = input("Answer (yes / no): ").strip().lower()
                is_yes = ans in ("yes", "y")

                for c in scores:
                    p = G.edges[c, parent]["p_e_given_c"] if G.has_edge(c, parent) else SMOOTH

                    if is_yes:
                        delta = safe_log(p)
                        observed_yes.add(parent)
                        scores[c] = capped_add(scores[c], delta)
                    else:
                        if p >= ABSENCE_PROB_THRESHOLD:
                            penalty = safe_log(1.0 - p)
                            scores[c] = capped_add(scores[c], ABSENCE_WEIGHT*penalty)
                        observed_no.add(parent)

                asked.add(parent)

            if parent in observed_no: # is asked but is not in observed_yes
                continue
            # if parent is asked and is in observed_yes, we proceed to ask evidence

        asked.add(evidence)

        print(f"\n🩺 Question: {G.nodes[evidence]['question_en']}")
        print(f"\nEvidence ID: {evidence}")
        values = [
            v for _, v in G.out_edges(evidence)
            if G.nodes[v]["type"].startswith("possible")
        ]

        # ---------------- Binary evidence ----------------
        if not values:
            ans = input("Answer (yes / no): ").strip().lower()
            is_yes = ans in ("yes", "y")

            for c in scores:
                p = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else SMOOTH

                if is_yes:
                    delta = safe_log(p)
                    observed_yes.add(evidence)
                    scores[c] = capped_add(scores[c], delta)
                else:
                    if p >= ABSENCE_PROB_THRESHOLD:
                        penalty = safe_log(1.0 - p)
                        scores[c] = capped_add(scores[c], ABSENCE_WEIGHT*penalty)
                    observed_no.add(evidence)

            continue


        # ---------------- Value-based evidence ----------------
        # shown_values = get_top_values(G, evidence, candidate_conditions, top_k_values)

        # print("\nOptions:")
        # for i, v in enumerate(shown_values):
        #     print(f"  {i}: {G.nodes[v]['value']}")
        # print(f"  {len(shown_values)}: Other / none")

        # choice = input(f"Choose options (0-{len(shown_values)}, space separated): ")
        choice_text = input()
        choice_text_with_evidence = G.nodes[evidence]['question_en'] + " " + choice_text
        matches = nlu._find_best_match(choice_text_with_evidence)
        print(f"NLU Match: {matches}")
        if matches:
            for match in matches:
                if match and match['value']:
                    if isinstance(match['value'], list):
                        chosen_values = match['value']
                        print(f"Chosen values: {chosen_values}")
                        observed_yes.add(evidence)
                        for chosen_value in chosen_values:
                            

                            for c in scores:
                                best_pv = SMOOTH

                                stats = G.edges[evidence, chosen_value].get("cond_stats", {})
                                best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", SMOOTH))

                                scores[c] = capped_add(scores[c], safe_log(best_pv))
                    else:
                        chosen_value = match['value']
                        print(f"Chosen value: {chosen_value}")
                        observed_yes.add(evidence)

                        for c in scores:
                            best_pv = SMOOTH

                            stats = G.edges[evidence, chosen_value].get("cond_stats", {})
                            best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", SMOOTH))

                            scores[c] = capped_add(scores[c], safe_log(best_pv))
                    continue

        # display(choice)
        # choices = choice.split()
        # if not all(ch.isdigit() for ch in choices):
        #     continue

        # choices = [int(ch) for ch in choices]
        # if any(ch == len(shown_values) for ch in choices):
        #     observed_no.add(evidence)
        #     continue

        # if any(ch < 0 or ch >= len(shown_values) for ch in choices):
        #     continue

        # observed_yes.add(evidence)
        # chosen_values = [shown_values[ch] for ch in choices]

        # for c in scores:
        #     best_pv = SMOOTH

        #     for chosen_value in chosen_values:
        #         stats = G.edges[evidence, chosen_value].get("cond_stats", {})
        #         best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", SMOOTH))

        #     scores[c] = capped_add(scores[c], safe_log(best_pv))

    print("\n=== FINAL RANKED CONDITIONS ===")
    for c, s in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(f"{c:40s} score={s:.4f}")



In [6]:
import math

SMOOTH = 0.01


def safe_log(p):
    return math.log(max(p, SMOOTH))


def find_value_node(G, evidence, value_text):
    """
    Find a value node connected to an evidence by matching its text.
    """
    for _, v in G.out_edges(evidence):
        if G.nodes[v]["value"] == value_text:
            return v
    return None


def apply_initial_evidence(G, scores, evidences, values):
    """
    Initial evidence rules:
    - Binary evidence: if present → YES, if missing → UNKNOWN
    - Multivalued evidence: always present with a value
    - No explicit NO at initialization

    Returns:
    - initial_asked: set of evidence IDs that should be treated as already asked
    """

    initial_asked = set()
    observed_yes = set()
    observed_no = set()

    # Pass 1: record explicit presence + infer parents
    for e, v in zip(evidences, values):
        initial_asked.add(e)

        # If child value present, parent is implicitly present
        parent = G.nodes[e].get("parent")
        print(f"Evidence: {e}, Value: {v}, Parent: {parent}")
        if v not in ['YES','NO'] and parent:
            initial_asked.add(parent)
        if v == "YES":
            observed_yes.add(e)
        elif v == "NO":
            observed_no.add(e)

    # Pass 2: apply evidence to scores
    for evid, values in zip(evidences, values):
        # ----------------------------------
        # Multivalued / categorical evidence
        # ----------------------------------
        if values not in ['YES','NO']:
            for v in values:
                for cond in scores:
                    stats = G.edges[evid, v].get("cond_stats", {})
                    p_v = stats.get(cond, {}).get("p_v_given_e_c", 0.0)
                    scores[cond] += safe_log(p_v)
            continue

        # -----------------------------
        # Binary evidence (YES)
        # -----------------------------
        if values == "YES":
            for cond in scores:
                p = (
                    G.edges[cond, evid].get("p_e_given_c", 0.0)
                    if G.has_edge(cond, evid)
                    else 0.0
                )
                scores[cond] += safe_log(p)
        else:
            for cond in scores:
                p = (
                    G.edges[cond, evid].get("p_e_given_c", 0.0)
                    if G.has_edge(cond, evid)
                    else 0.0
                )
                if p >= ABSENCE_PROB_THRESHOLD:
                    penalty = safe_log(1.0 - p)
                    scores[cond] = capped_add(scores[cond], ABSENCE_WEIGHT*penalty)

    return initial_asked, observed_yes, observed_no



# Initialize condition scores (uniform prior)
# scores = {c: 0.0 for c in cond_count}

# Example 1: Binary symptom present (e.g., fever)
# Replace E_XX with the actual fever evidence ID in your dataset
# apply_initial_evidence(G, scores, evidence="E_XX", value=None)

# Example 2: Value-based symptom (e.g., pain location = lower chest)
# evidence_id = ["E_133"]  # pain location evidence
# value_text = ["lower chest"]

# value_node = find_value_node(G, evidence_id, value_text)
# if value_node is not None:
#     apply_initial_evidence(G, scores, evidence=evidence_id, value=value_node)


In [ ]:
class KG_Traversal(DDxGraphNLU):
    # ---------------- CONFIG ----------------
    SMOOTH = 1e-6
    MAX_DELTA = 2.0
    ABSENCE_PROB_THRESHOLD = 0.5
    ABSENCE_WEIGHT = 0.5

    def __init__(
        self,
        G,
        scores,
        initial_asked=None,
        observed_yes=None,
        observed_no=None
    ):
        super().__init__(G)  # DDxGraphNLU init

        self.G = G
        self.scores = scores

        self.asked = set() if initial_asked is None else set(initial_asked)
        self.observed_yes = set() if observed_yes is None else set(observed_yes)
        self.observed_no = set() if observed_no is None else set(observed_no)
    
    def safe_log(self, p):
        """Calculates natural log while avoiding mathematical errors with zero."""
        return math.log(max(p, self.SMOOTH))

    def capped_add(self, score, delta):
        """Updates a condition's score but prevents outliers from swinging the score too far."""
        return score + max(-self.MAX_DELTA, min(self.MAX_DELTA, delta))

    def get_discriminating_evidence(self, G, candidate_conditions, scores, asked):
        """
        Selects the symptom that best distinguishes between the current top disease candidates.
        Uses a weighted variance approach (similar to Information Gain).
        """
        best_e, best_gain = None, -1.0

        # 1. Convert current logarithmic scores into a normalized probability distribution (Posteriors)
        max_s = max(scores[c] for c in candidate_conditions)
        post = {
            c: math.exp(scores[c] - max_s)
            for c in candidate_conditions
        }
        Z = sum(post.values())
        post = {c: p / Z for c, p in post.items()} # Ensure they sum to 1.0

        # 2. Iterate through all possible symptoms connected to the top candidates
        for c in candidate_conditions:
            for _, e, _ in G.out_edges(c, data=True):
                if G.nodes[e]["type"] != "evidence" or e in asked:
                    continue

                # 3. Calculate how much 'disagreement' exists between candidates for this symptom
                ps = []
                for c2 in candidate_conditions:
                    p = G.edges[c2, e]["p_e_given_c"] if G.has_edge(c2, e) else self.SMOOTH
                    ps.append((post[c2], p))

                mean = sum(w * p for w, p in ps)
                gain = sum(w * (p - mean) ** 2 for w, p in ps)

                # 4. Keep the symptom that provides the most 'Information Gain'
                if gain > best_gain:
                    best_gain, best_e = gain, e

        return best_e

    def run_ddx_traversal(self, G, scores, max_steps=15, top_k_conditions=10, top_k_values=20):
        """
        Conducts the interactive diagnostic session.
        """        
        asked = self.asked
        observed_yes = self.observed_yes
        observed_no = self.observed_no
        nlu = DDxGraphNLU(G)

        print("\n=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===")

        for step in range(max_steps):
            # --- A. Rank and Display the current Top Candidates ---
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_conditions]
            candidate_conditions = [c for c, _ in ranked]

            print(f"\nStep {step + 1} — Current Differential:")
            for c, s in ranked:
                print(f"  {c:40s} score={s:.4f}")

            if len(candidate_conditions) <= 1:
                break

            # --- B. Decide the next Question ---
            evidence = self.get_discriminating_evidence(G, candidate_conditions, scores, asked)
            if evidence is None: break

            print(f"Selected evidence: {G.nodes[evidence]}")
            parent = G.nodes[evidence].get("parent", None)
            print(f"Parent evidence: {parent}")

            if parent and parent != evidence and parent not in observed_yes:
                if parent not in asked:
                    print(f"\n🩺 Question: {G.nodes[parent]['question_en']}")
                    print(f"\nEvidence ID: {parent}")

                    ans = input("Answer (yes / no): ").strip().lower()
                    is_yes = ans in ("yes", "y")

                    for c in scores:
                        p = G.edges[c, parent]["p_e_given_c"] if G.has_edge(c, parent) else self.SMOOTH

                        if is_yes:
                            delta = self.safe_log(p)
                            observed_yes.add(parent)
                            scores[c] = self.capped_add(scores[c], delta)
                        else:
                            if p >= self.ABSENCE_PROB_THRESHOLD:
                                penalty = self.safe_log(1.0 - p)
                                scores[c] = self.capped_add(scores[c], self.ABSENCE_WEIGHT*penalty)
                            observed_no.add(parent)

                    asked.add(parent)

                if parent in observed_no: # is asked but is not in observed_yes
                    continue
                # if parent is asked and is in observed_yes, we proceed to ask evidence

            asked.add(evidence)

            print(f"\n🩺 Question: {G.nodes[evidence]['question_en']}")
            print(f"\nEvidence ID: {evidence}")
            values = [
                v for _, v in G.out_edges(evidence)
                if G.nodes[v]["type"].startswith("possible")
            ]

            # ---------------- Binary evidence ----------------
            if not values:
                ans = input("Answer (yes / no): ").strip().lower()
                is_yes = ans in ("yes", "y")

                for c in scores:
                    p = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else self.SMOOTH

                    if is_yes:
                        delta = self.safe_log(p)
                        observed_yes.add(evidence)
                        scores[c] = self.capped_add(scores[c], delta)
                    else:
                        if p >= self.ABSENCE_PROB_THRESHOLD:
                            penalty = self.safe_log(1.0 - p)
                            scores[c] = self.capped_add(scores[c], self.ABSENCE_WEIGHT*penalty)
                        observed_no.add(evidence)

                continue

            choice_text = input()
            choice_text_with_evidence = G.nodes[evidence]['question_en'] + " " + choice_text
            matches = nlu._find_best_match(choice_text_with_evidence)
            print(f"NLU Match: {matches}")
            if matches:
                for match in matches:
                    if match and match['value']:
                        if isinstance(match['value'], list):
                            chosen_values = match['value']
                            print(f"Chosen values: {chosen_values}")
                            observed_yes.add(evidence)
                            for chosen_value in chosen_values:
                                

                                for c in scores:
                                    best_pv = self.SMOOTH

                                    stats = G.edges[evidence, chosen_value].get("cond_stats", {})
                                    best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH))

                                    scores[c] = self.capped_add(scores[c], self.safe_log(best_pv))
                        else:
                            chosen_value = match['value']
                            print(f"Chosen value: {chosen_value}")
                            observed_yes.add(evidence)

                            for c in scores:
                                best_pv = self.SMOOTH

                                stats = G.edges[evidence, chosen_value].get("cond_stats", {})
                                best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH))

                                scores[c] = self.capped_add(scores[c], self.safe_log(best_pv))
                        continue

        print("\n=== FINAL RANKED CONDITIONS ===")
        for c, s in sorted(scores.items(), key=lambda x: x[1], reverse=True):
            print(f"{c:40s} score={s:.4f}")

    def apply_initial_evidence(self, G, scores, evidences, values):
        """
        Initial evidence rules:
        - Binary evidence: if present → YES, if missing → UNKNOWN
        - Multivalued evidence: always present with a value
        - No explicit NO at initialization

        Returns:
        - initial_asked: set of evidence IDs that should be treated as already asked
        """

        initial_asked = set()
        observed_yes = set()
        observed_no = set()

        # Pass 1: record explicit presence + infer parents
        for e, v in zip(evidences, values):
            initial_asked.add(e)

            # If child value present, parent is implicitly present
            parent = G.nodes[e].get("parent")
            print(f"Evidence: {e}, Value: {v}, Parent: {parent}")
            if v not in ['YES','NO'] and parent:
                initial_asked.add(parent)
            if v == "YES":
                observed_yes.add(e)
            elif v == "NO":
                observed_no.add(e)

        # Pass 2: apply evidence to scores
        for evid, values in zip(evidences, values):
            # ----------------------------------
            # Multivalued / categorical evidence
            # ----------------------------------
            if values not in ['YES','NO']:
                for v in values:
                    for cond in scores:
                        stats = G.edges[evid, v].get("cond_stats", {})
                        p_v = stats.get(cond, {}).get("p_v_given_e_c", 0.0)
                        scores[cond] += self.safe_log(p_v)
                continue

            # -----------------------------
            # Binary evidence (YES)
            # -----------------------------
            if values == "YES":
                for cond in scores:
                    p = (
                        G.edges[cond, evid].get("p_e_given_c", 0.0)
                        if G.has_edge(cond, evid)
                        else 0.0
                    )
                    scores[cond] += self.safe_log(p)
            else:
                for cond in scores:
                    p = (
                        G.edges[cond, evid].get("p_e_given_c", 0.0)
                        if G.has_edge(cond, evid)
                        else 0.0
                    )
                    if p >= ABSENCE_PROB_THRESHOLD:
                        penalty = self.safe_log(1.0 - p)
                        scores[cond] = self.capped_add(scores[cond], ABSENCE_WEIGHT*penalty)

        return initial_asked, observed_yes, observed_no

            

In [ ]:
import math

class KG_Traversal(DDxGraphNLU):
    # ---------------- CONFIG ----------------
    SMOOTH = 1e-6
    MAX_DELTA = 2.0
    ABSENCE_PROB_THRESHOLD = 0.5
    ABSENCE_WEIGHT = 0.5

    def __init__(self, G, scores, user_input=None):
        super().__init__(G)

        self.G = G
        self.scores = scores

        self.asked = set()
        self.observed_yes = set()
        self.observed_no = set()
    
        if user_input:
            self._parse_initial_query(user_input)

    def _parse_initial_query(self, user_input):
        """
        Parse free-text user input and initialize evidence + scores.
        """
        results, evidences, values = self.parse_query(user_input)

        if not evidences:
            return

        print("\nParsed initial evidence:")
        for e, v in zip(evidences, values):
            print(f"  {e} → {v}")

        self.apply_initial_evidence(
            evidences=evidences,
            values=values
        )


    # ---------------- UTIL ----------------
    def safe_log(self, p):
        return math.log(max(p, self.SMOOTH))

    def capped_add(self, score, delta):
        return score + max(-self.MAX_DELTA, min(self.MAX_DELTA, delta))

    # ---------------- EVIDENCE SELECTION ----------------
    def get_discriminating_evidence(self, candidate_conditions):
        best_e, best_gain = None, -1.0

        max_s = max(self.scores[c] for c in candidate_conditions)
        post = {
            c: math.exp(self.scores[c] - max_s)
            for c in candidate_conditions
        }
        Z = sum(post.values())
        post = {c: p / Z for c, p in post.items()}

        for c in candidate_conditions:
            for _, e, _ in self.G.out_edges(c, data=True):
                if self.G.nodes[e]["type"] != "evidence" or e in self.asked:
                    continue

                ps = []
                for c2 in candidate_conditions:
                    p = (
                        self.G.edges[c2, e]["p_e_given_c"]
                        if self.G.has_edge(c2, e)
                        else self.SMOOTH
                    )
                    ps.append((post[c2], p))

                mean = sum(w * p for w, p in ps)
                gain = sum(w * (p - mean) ** 2 for w, p in ps)

                if gain > best_gain:
                    best_gain, best_e = gain, e

        return best_e

    # ---------------- SCORE UPDATES ----------------
    def apply_binary_answer(self, evidence, is_yes):
        for c in self.scores:
            p = (
                self.G.edges[c, evidence]["p_e_given_c"]
                if self.G.has_edge(c, evidence)
                else self.SMOOTH
            )

            if is_yes:
                self.scores[c] = self.capped_add(self.scores[c], self.safe_log(p))
                self.observed_yes.add(evidence)
            else:
                if p >= self.ABSENCE_PROB_THRESHOLD:
                    penalty = self.safe_log(1.0 - p)
                    self.scores[c] = self.capped_add(
                        self.scores[c],
                        self.ABSENCE_WEIGHT * penalty
                    )
                self.observed_no.add(evidence)

        self.asked.add(evidence)

    def apply_value_answer(self, evidence, chosen_values):
        self.observed_yes.add(evidence)

        for c in self.scores:
            best_pv = self.SMOOTH
            for v in chosen_values:
                stats = self.G.edges[evidence, v].get("cond_stats", {})
                best_pv = max(
                    best_pv,
                    stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH)
                )

            self.scores[c] = self.capped_add(self.scores[c], self.safe_log(best_pv))

        self.asked.add(evidence)

    # ---------------- INITIAL EVIDENCE ----------------
    def apply_initial_evidence(self, evidences, values):
        for evid, val in zip(evidences, values):
            self.asked.add(evid)

            parent = self.G.nodes[evid].get("parent")
            if val not in ("YES", "NO") and parent:
                self.asked.add(parent)

            if val == "YES":
                self.observed_yes.add(evid)
            elif val == "NO":
                self.observed_no.add(evid)

            # Apply scoring
            if val not in ("YES", "NO"):
                for v in val:
                    for c in self.scores:
                        stats = self.G.edges[evid, v].get("cond_stats", {})
                        p = stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH)
                        self.scores[c] += self.safe_log(p)
            else:
                self.apply_binary_answer(evid, val == "YES")

    # ---------------- MAIN LOOP ----------------
    def run(self, max_steps=5, top_k_conditions=10):
        print("\n=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===")

        # for step in range(max_steps):
        #     ranked = sorted(
        #         self.scores.items(),
        #         key=lambda x: x[1],
        #         reverse=True
        #     )[:top_k_conditions]

        #     candidates = [c for c, _ in ranked]

        #     print(f"\nStep {step + 1}")
        #     for c, s in ranked:
        #         print(f"{c:40s} score={s:.4f}")

        #     if len(candidates) <= 1:
        #         break

        #     evidence = self.get_discriminating_evidence(candidates)
        #     if not evidence:
        #         break

        #     print(f"\n🩺 {self.G.nodes[evidence]['question_en']}")

        #     values = [
        #         v for _, v in self.G.out_edges(evidence)
        #         if self.G.nodes[v]["type"].startswith("possible")
        #     ]

        #     if not values:
        #         ans = input("yes / no: ").strip().lower()
        #         self.apply_binary_answer(evidence, ans in ("y", "yes"))
        #         continue

        #     text = input()
        #     query = self.G.nodes[evidence]['question_en'] + " " + text
        #     matches = self._find_best_match(query)

        #     if matches:
        #         for m in matches:
        #             if m and m["value"]:
        #                 chosen = m["value"] if isinstance(m["value"], list) else [m["value"]]
        #                 self.apply_value_answer(evidence, chosen)

        # print("\n=== FINAL RANKED CONDITIONS ===")
        # for c, s in sorted(self.scores.items(), key=lambda x: x[1], reverse=True):
        #     print(f"{c:40s} score={s:.4f}")

        for step in range(max_steps):
            # --- A. Rank and Display the current Top Candidates ---
            ranked = sorted(self.scores.items(), key=lambda x: x[1], reverse=True)[:top_k_conditions]
            candidate_conditions = [c for c, _ in ranked]

            print(f"\nStep {step + 1} — Current Differential:")
            for c, s in ranked:
                print(f"  {c:40s} score={s:.4f}")

            if len(candidate_conditions) <= 1:
                break

            # --- B. Decide the next Question ---
            evidence = self.get_discriminating_evidence(candidate_conditions)
            if evidence is None: break

            print(f"Selected evidence: {self.G.nodes[evidence]}")
            parent = self.G.nodes[evidence].get("parent", None)
            print(f"Parent evidence: {parent}")

            if parent and parent != evidence and parent not in self.observed_yes:
                if parent not in self.asked:
                    print(f"\n🩺 Question: {self.G.nodes[parent]['question_en']}")
                    print(f"\nEvidence ID: {parent}")

                    ans = input("Answer (yes / no): ").strip().lower()
                    is_yes = ans in ("yes", "y")

                    for c in self.scores:
                        p = self.G.edges[c, parent]["p_e_given_c"] if self.G.has_edge(c, parent) else self.SMOOTH

                        if is_yes:
                            delta = self.safe_log(p)
                            self.observed_yes.add(parent)
                            self.scores[c] = self.capped_add(self.scores[c], delta)
                        else:
                            if p >= self.ABSENCE_PROB_THRESHOLD:
                                penalty = self.safe_log(1.0 - p)
                                self.scores[c] = self.capped_add(self.scores[c], self.ABSENCE_WEIGHT*penalty)
                            self.observed_no.add(parent)

                    self.asked.add(parent)

                if parent in self.observed_no: # is asked but is not in observed_yes
                    continue
                # if parent is asked and is in observed_yes, we proceed to ask evidence

            self.asked.add(evidence)
            print(f"\n🩺 Question: {self.G.nodes[evidence]['question_en']}")
            print(f"\nEvidence ID: {evidence}")
            values = [
                v for _, v in self.G.out_edges(evidence)
                if self.G.nodes[v]["type"].startswith("possible")
            ]

            # ---------------- Binary evidence ----------------
            if not values:
                ans = input("Answer (yes / no): ").strip().lower()
                is_yes = ans in ("yes", "y")

                for c in self.scores:
                    p = self.G.edges[c, evidence]["p_e_given_c"] if self.G.has_edge(c, evidence) else self.SMOOTH

                    if is_yes:
                        delta = self.safe_log(p)
                        self.observed_yes.add(evidence)
                        self.scores[c] = self.capped_add(self.scores[c], delta)
                    else:
                        if p >= self.ABSENCE_PROB_THRESHOLD:
                            penalty = self.safe_log(1.0 - p)
                            self.scores[c] = self.capped_add(self.scores[c], self.ABSENCE_WEIGHT*penalty)
                        self.observed_no.add(evidence)

                continue

            choice_text = input()
            choice_text_with_evidence = self.G.nodes[evidence]['question_en'] + " " + choice_text
            matches = nlu._find_best_match(choice_text_with_evidence)
            print(f"NLU Match: {matches}")
            if matches:
                for match in matches:
                    if match and match['value']:
                        if isinstance(match['value'], list):
                            chosen_values = match['value']
                            print(f"Chosen values: {chosen_values}")
                            self.observed_yes.add(evidence)
                            for chosen_value in chosen_values:
                                

                                for c in self.scores:
                                    best_pv = self.SMOOTH

                                    stats = self.G.edges[evidence, chosen_value].get("cond_stats", {})
                                    best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH))

                                    self.scores[c] = self.capped_add(self.scores[c], self.safe_log(best_pv))
                        else:
                            chosen_value = match['value']
                            print(f"Chosen value: {chosen_value}")
                            self.observed_yes.add(evidence)
                            for c in self.scores:
                                best_pv = self.SMOOTH

                                stats = self.G.edges[evidence, chosen_value].get("cond_stats", {})
                                best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", self.SMOOTH))

                                self.scores[c] = self.capped_add(self.scores[c], self.safe_log(best_pv))
                        continue

        print("\n=== FINAL RANKED CONDITIONS ===")
        for c, s in sorted(self.scores.items(), key=lambda x: x[1], reverse=True):
            print(f"{c:40s} score={s:.4f}")


In [16]:
user_input = input("Describe your symptoms: ")

scores = {c: 0.0 for c in cond_count}

traversal = KG_Traversal(
    G,
    scores,
    user_input=user_input
)

traversal.run()

Loading Model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext...


No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Loading embeddings from Graph Evidence Nodes...
 -> Loaded 223 evidence vectors.

[NLU] Processing: 'i have cough'
 -> Chunk 1: 'i have cough'
    [Debug] Checking Top 5 Candidates for span: 'i have cough'
      -> Candidate E_201: RawQ=0.925, FinalScore=0.925
      -> Candidate E_202: RawQ=0.738, FinalScore=0.738
      -> Candidate E_45: RawQ=0.721, FinalScore=0.721
      -> Candidate E_77: RawQ=0.633, FinalScore=0.633
      -> Candidate E_166: RawQ=0.619, FinalScore=0.619

Parsed initial evidence:
  E_201 → YES

=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===

Step 1 — Current Differential:
  Tuberculosis                             score=-0.0001
  Bronchitis                               score=-0.1372
  Chronic rhinosinusitis                   score=-0.1856
  Pneumonia                                score=-0.1888
  Influenza                                score=-0.1970
  Acute COPD exacerbation / infection      score=-0.2543
  Acute rhinosinusitis                     score=-0.2561
  Bronch

In [11]:
# 1. Initialize scores
scores = {c: 0.0 for c in cond_count}
# 30,F,Pneumonia,"['E_201': 1, 'E_66': 1, 'E_91': 1, 'E_94': 1, 'E_144': 1, 'E_175': 1, 'E_53': 1, 'E_55': 'V_101', 'E_124': 1, 'E_79': 1, 'E_214': 1, 'E_204': 'V_4']",E_201
# 55,M,peripheral artery disease (PAD),"['E_53': 1, 'E_55': 'V_149', 'E_93': 1, 'E_218': 1, 'E_216': 1, 'E_69': 1, 'E_79': 1, 'E_104': 1, 'E_108': 1]",E_53
# 55,M,Bronchitis,"['E_66': 1, 'E_201': 1, 'E_53': 1, 'E_54': 'V_181', 'E_55': ['V_55', 'V_56', 'V_101', 'V_29'], 'E_57': 1, 'E_89': 1, 'E_79': 1, 'E_123': 1, 'E_204': 1]",E_201
# 62,F,Acute COPD exacerbation / infection,"['E_201': 1, 'E_77': 1, 'E_66': 1, 'E_89': 1, 'E_79': 1, 'E_104': 1]",E_201
# 52,M,Inguinal hernia,"['E_53', 'E_54_@_V_183', 'E_55_@_V_87', 'E_55_@_V_88', 'E_55_@_V_99', 'E_55_@_V_100', 'E_55_@_V_169', 'E_56_@_1', 'E_57_@_V_123', 'E_58_@_6', 'E_59_@_3', 'E_129', 'E_130_@_V_138', 'E_131_@_V_10', 'E_132_@_5', 'E_133_@_V_88', 'E_134_@_4', 'E_135_@_V_12', 'E_136_@_0', 'E_160', 'E_203', 'E_204_@_V_10', 'E_221']",E_53
# 4,,M,Pulmonary embolism,"['E_45', 'E_53', 'E_54_@_V_161', 'E_54_@_V_192', 'E_55_@_V_55', 'E_55_@_V_56', 'E_55_@_V_127', 'E_55_@_V_128', 'E_55_@_V_170', 'E_56_@_5', 'E_57_@_V_55', 'E_57_@_V_127', 'E_57_@_V_128', 
# 'E_57_@_V_170', 'E_57_@_V_171', 'E_58_@_5', 'E_59_@_10', 'E_66', 'E_109', 'E_110', 'E_152_@_V_123', 'E_159', 'E_196', 'E_204_@_V_4', 'E_220']",E_45

# 2. Apply initial evidence
nlu = DDxGraphNLU(G)

    # 2. Define a complex user query with multiple symptoms and negation
user_input = input()

# 3. Run the Parser
results, evidence, values = nlu.parse_query(user_input)
# evidence = ["E_55"]
# values = ["V_29"]

initial_asked, observed_yes, observed_no = apply_initial_evidence(G, scores, evidences=evidence, values=values)

# 3. Run traversal WITH scores
run_ddx_traversal_1(G, scores, max_steps=5, initial_asked=initial_asked, observed_yes=observed_yes, observed_no=observed_no)


Loading Model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext...


No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Loading embeddings from Graph Evidence Nodes...
 -> Loaded 223 evidence vectors.

[NLU] Processing: ''
Loading Model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext...


No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Loading embeddings from Graph Evidence Nodes...
 -> Loaded 223 evidence vectors.

=== INTERACTIVE DIAGNOSTIC TRAVERSAL ===

Step 1 — Current Differential:
  URTI                                     score=0.0000
  HIV (initial infection)                  score=0.0000
  Pneumonia                                score=0.0000
  Chronic rhinosinusitis                   score=0.0000
  Viral pharyngitis                        score=0.0000
  Anemia                                   score=0.0000
  Atrial fibrillation                      score=0.0000
  Allergic sinusitis                       score=0.0000
  Larygospasm                              score=0.0000
  Cluster headache                         score=0.0000
Selected evidence: {'type': 'evidence', 'question_en': 'Do you feel pain somewhere?', 'is_antecedent': False, 'data_type': 'M', 'parent': 'E_53', 'embedding': [0.48802056908607483, -0.3902725577354431, 0.20318228006362915, 0.24378520250320435, -0.7901895046234131, -0.1913250982761383,

In [ ]:
import math
import re

# ---------------- CONFIG ----------------
SMOOTH = 1e-6
MAX_DELTA = 2.0
ABSENCE_PROB_THRESHOLD = 0.5
ABSENCE_WEIGHT = 0.5

# ---------------- UTILS ----------------
def safe_log(p):
    return math.log(max(p, SMOOTH))

def capped_add(score, delta):
    return score + max(-MAX_DELTA, min(MAX_DELTA, delta))

# ---------------- VALUE SELECTION ----------------
def get_top_values(G, evidence, candidate_conditions):
    scores = []
    for _, v in G.out_edges(evidence):
        if not G.nodes[v]["type"].startswith("possible"):
            continue
        stats = G.edges[evidence, v].get("cond_stats", {})
        max_p = max(
            stats.get(c, {}).get("p_v_given_e_c", 0.0)
            for c in candidate_conditions
        )
        scores.append((v, max_p))

    scores.sort(key=lambda x: x[1], reverse=True)
    return [v for v, _ in scores]

# Regex for DDXPlus value format
VALUE_PATTERN = re.compile(r'^(E_\d+)_@_(V_\d+|\d+)$')

def find_existing_values(evidences, shown_values, evidence):
    chosen = []
    for ev in evidences:
        m = VALUE_PATTERN.match(ev)
        if not m:
            continue
        eid, vid = m.group(1), m.group(2)
        if eid == evidence and vid in shown_values:
            chosen.append(vid)
    return chosen

# ---------------- DISCRIMINATING EVIDENCE ----------------
def get_discriminating_evidence(G, candidate_conditions, scores, asked):
    best_e, best_gain = None, -1.0

    max_s = max(scores[c] for c in candidate_conditions)
    post = {c: math.exp(scores[c] - max_s) for c in candidate_conditions}
    Z = sum(post.values())
    post = {c: p / Z for c, p in post.items()}

    for c in candidate_conditions:
        for _, e, _ in G.out_edges(c, data=True):
            if G.nodes[e]["type"] != "evidence" or e in asked:
                continue

            ps = []
            for c2 in candidate_conditions:
                p = G.edges[c2, e]["p_e_given_c"] if G.has_edge(c2, e) else SMOOTH
                ps.append((post[c2], p))

            mean = sum(w * p for w, p in ps)
            gain = sum(w * (p - mean) ** 2 for w, p in ps)

            if gain > best_gain:
                best_gain, best_e = gain, e

    return best_e

# ---------------- MAIN TESTING LOOP ----------------
def run_ddx_traversal_testing(
    G,
    scores,
    evidences,
    pathology,
    k=5,
    max_steps=5,
    initial_asked=None
):
    asked = set() if initial_asked is None else set(initial_asked)
    observed_yes = set()
    observed_no = set()

    # ---------------- Initialize observed YES from DDXPlus evidences ----------------
    for ev in evidences:
        if "_@_" in ev:
            observed_yes.add(ev.split("_@_")[0])
        else:
            observed_yes.add(ev)

    # ---------------- Traversal ----------------
    for step in range(max_steps):

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        candidate_conditions = [c for c, _ in ranked[:10]]

        if len(candidate_conditions) <= 1:
            break

        evidence = get_discriminating_evidence(G, candidate_conditions, scores, asked)
        if evidence is None:
            break

        parent = G.nodes[evidence].get("parent")

        # ---------- Parent gating ----------
        if parent and parent != evidence:
            if parent not in observed_yes and parent not in asked:
                is_yes = parent in observed_yes
                for c in scores:
                    p = G.edges[c, parent]["p_e_given_c"] if G.has_edge(c, parent) else SMOOTH
                    if is_yes:
                        scores[c] = capped_add(scores[c], safe_log(p))
                    else:
                        if p >= ABSENCE_PROB_THRESHOLD:
                            scores[c] = capped_add(scores[c], ABSENCE_WEIGHT * safe_log(1 - p))
                asked.add(parent)

            if parent in observed_no:
                continue

        asked.add(evidence)

        # ---------------- Binary evidence ----------------
        values = [
            v for _, v in G.out_edges(evidence)
            if G.nodes[v]["type"].startswith("possible")
        ]

        if not values:
            is_yes = evidence in observed_yes
            for c in scores:
                p = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else SMOOTH
                if is_yes:
                    scores[c] = capped_add(scores[c], safe_log(p))
                else:
                    if p >= ABSENCE_PROB_THRESHOLD:
                        scores[c] = capped_add(scores[c], ABSENCE_WEIGHT * safe_log(1 - p))
            continue

        # ---------------- Value-based evidence ----------------
        shown_values = get_top_values(G, evidence, candidate_conditions)
        chosen_values = find_existing_values(evidences, shown_values, evidence)

        if not chosen_values:
            observed_no.add(evidence)
            for c in scores:
                p = G.edges[c, evidence]["p_e_given_c"] if G.has_edge(c, evidence) else SMOOTH
                if p >= ABSENCE_PROB_THRESHOLD:
                    scores[c] = capped_add(scores[c], ABSENCE_WEIGHT * safe_log(1 - p))
            continue

        observed_yes.add(evidence)
        for c in scores:
            best_pv = SMOOTH
            for v in chosen_values:
                stats = G.edges[evidence, v].get("cond_stats", {})
                best_pv = max(best_pv, stats.get(c, {}).get("p_v_given_e_c", SMOOTH))
            scores[c] = capped_add(scores[c], safe_log(best_pv))

    # ---------------- Evaluation ----------------
    final_ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_k = {c for c, _ in final_ranked[:k]}
    return pathology in top_k


In [ ]:
test_patients =pd.read_csv(".././Data/ddxplus/release_test_patients.csv")
test_patients_sample = test_patients.sample(n=10000, random_state=42).reset_index(drop=True)
all_ks = [1,3,5,7]
max_steps = [4,6,8,10]
accuracies = {}

for step in max_steps:
    for k in all_ks:
        total_hits = 0
        total_tests = 0
        for idx, row in test_patients_sample.iterrows():
            init_evid = row["INITIAL_EVIDENCE"]
            match = pattern.match(init_evid)
            initial_evidences = []
            initial_values = []
            if match:
                initial_evidences.append(match.group("eid"))
                initial_values.append(match.group("vid"))
            else:
                initial_evidences.append(init_evid)
                initial_values.append(None)
            
            scores = {c: 0.0 for c in cond_count}
            initial_asked = apply_initial_evidence(
                G,
                scores,
                evidences=initial_evidences,
                values=initial_values
            )

            hit = run_ddx_traversal_testing(
                G,
                scores,
                evidences=[init_evid] + ast.literal_eval(row["EVIDENCES"]),
                pathology=row["PATHOLOGY"],
                k=k,  
                max_steps=step,
                initial_asked=initial_asked
            )
            total_tests += 1
            if hit:
                total_hits += 1

        accuracy = total_hits / total_tests
        accuracies[(k, step)] = accuracy
        print(f"Top-{k} accuracy with max_steps={step}: {accuracy:.4f}")



Top-1 accuracy with max_steps=4: 0.7868
Top-3 accuracy with max_steps=4: 0.8857
Top-5 accuracy with max_steps=4: 0.9156
Top-7 accuracy with max_steps=4: 0.9400
Top-1 accuracy with max_steps=6: 0.7832
Top-3 accuracy with max_steps=6: 0.8830
Top-5 accuracy with max_steps=6: 0.9297


KeyboardInterrupt: 